In [ ]:
!pip -q install timm==1.0.11 transformers==4.45.2 peft==0.13.2 \
                accelerate==1.0.1 scikit-learn==1.5.2 \
                google-generativeai==0.8.3 optuna==3.6.1 \
                opencv-python-headless==4.10.0.84 grad-cam==1.5.4

In [ ]:
import os, json, random, time, copy
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ 본인 드라이브 구조에 맞게 수정
DRIVE_ROOT = Path('/content/drive/MyDrive/404:NOT HUMAN FOUND/data/test_Data_Woochul/test_dataset')   # 데이터셋 루트
AI_DIR     = DRIVE_ROOT / 'ai'        # 200개 AI 이미지
REAL_DIR   = DRIVE_ROOT / 'real'      # 100개 실제 이미지

WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
OUT  = WORK / 'outputs';      OUT.mkdir(exist_ok=True)
CKPT = Path('/content/drive/MyDrive/ai_image_ckpts'); CKPT.mkdir(exist_ok=True, parents=True)

IMG_EXT  = {'.jpg','.jpeg','.png','.webp','.bmp'}
IMG_SIZE = 224
BATCH    = 16
NUM_WORKERS = 2
EPOCHS   = 10            # 사용자 요청: 10 epoch
LR_HEAD  = 5e-4
LR_LORA  = 1e-4
WD       = 1e-4

print(f'AI dir   : {AI_DIR}  (존재: {AI_DIR.exists()})')
print(f'Real dir : {REAL_DIR} (존재: {REAL_DIR.exists()})')

In [ ]:
def list_images(folder):
    return [p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXT]

ai_imgs   = list_images(AI_DIR)
real_imgs = list_images(REAL_DIR)
print(f'✅ AI 이미지   : {len(ai_imgs)}개')
print(f'✅ 실제 이미지 : {len(real_imgs)}개')
print(f'⚠️ 클래스 불균형 비율 (AI:Real) = {len(ai_imgs)}:{len(real_imgs)}')

In [ ]:
def collect_meta(paths, label):
    rows = []
    for p in paths:
        try:
            with Image.open(p) as im:
                w, h = im.size
                mode = im.mode
            rows.append({'path': str(p), 'label': label,
                         'width': w, 'height': h, 'aspect': w/h,
                         'size_kb': p.stat().st_size/1024,
                         'format': p.suffix.lower(), 'mode': mode})
        except Exception as e:
            print(f'⚠️ {p.name}: {e}')
    return rows

meta = pd.DataFrame(collect_meta(ai_imgs, 1) + collect_meta(real_imgs, 0))
print('=== 기본 통계 ===')
print(meta.groupby('label').agg(
    n=('path','count'),
    mean_w=('width','mean'),  mean_h=('height','mean'),
    mean_kb=('size_kb','mean'), mean_aspect=('aspect','mean')
).round(1))
print('\n=== 포맷 분포 ===')
print(pd.crosstab(meta['label'].map({0:'real',1:'ai'}), meta['format']))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))

for lbl, name, color in [(0,'real','steelblue'),(1,'ai','coral')]:
    sub = meta[meta['label']==lbl]
    ax[0,0].hist(sub['width'],   alpha=0.6, label=name, color=color, bins=20)
    ax[0,1].hist(sub['height'],  alpha=0.6, label=name, color=color, bins=20)
    ax[0,2].hist(sub['size_kb'], alpha=0.6, label=name, color=color, bins=20)
ax[0,0].set_title('Width');   ax[0,0].legend()
ax[0,1].set_title('Height');  ax[0,1].legend()
ax[0,2].set_title('Size(KB)'); ax[0,2].legend()

# 샘플 이미지
for i in range(3):
    ax[1,i].imshow(Image.open(ai_imgs[i]).resize((200,200)))
    ax[1,i].set_title(f'AI sample {i+1}'); ax[1,i].axis('off')
plt.tight_layout(); plt.savefig(OUT/'eda.png', dpi=120); plt.show()

 Phase 3 — 전처리 (Train/Val/Test Split + Augmentation)

In [ ]:
# 전체 300장 → Train 70% / Val 15% / Test 15%
train_meta, tmp_meta = train_test_split(meta, test_size=0.30, stratify=meta['label'], random_state=SEED)
val_meta, test_meta  = train_test_split(tmp_meta, test_size=0.50, stratify=tmp_meta['label'], random_state=SEED)
print(f'Train {len(train_meta)} | Val {len(val_meta)} | Test {len(test_meta)}')
print('Train 분포:', train_meta['label'].value_counts().to_dict())

In [ ]:
from torchvision import transforms

CLIP_MEAN = [0.48145466, 0.4578275,  0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

In [ ]:
class ImgDataset(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['path']).convert('RGB')
        return self.tf(img), int(r['label']), r['path']

In [ ]:
# 클래스별 가중치 → 적은 쪽(real)을 더 자주 뽑게
counts = Counter(train_meta['label'])
weights = train_meta['label'].map(lambda y: 1.0/counts[y]).values
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(ImgDataset(train_meta, train_tf), batch_size=BATCH,
                          sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(ImgDataset(val_meta,   eval_tf), batch_size=BATCH,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(ImgDataset(test_meta,  eval_tf), batch_size=BATCH,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print('Loaders ready.')

In [ ]:
candidates = pd.DataFrame([
    {'Model':'CLIP-ViT-B/16 + LoRA',   'Inspired':'OmniAID + SIDA', 'Params':'~149M (LoRA: ~0.5M)', 'Best for':'본 워크플로우 ⭐'},
    {'Model':'ViT-Base + Linear Head', 'Inspired':'Baseline',       'Params':'~86M',                'Best for':'빠른 비교'},
    {'Model':'LLaVA + LoRA',           'Inspired':'SIDA',           'Params':'~7B',                  'Best for':'Colab Pro+ 필요'},
])
print(candidates.to_string(index=False))
print('\n→ 채택: CLIP-ViT-B/16 + LoRA (OmniAID의 CLIP-ViT 백본 + SIDA의 LoRA 결합)')

In [ ]:
class CLIPDetector(nn.Module):
    def __init__(self, base='openai/clip-vit-base-patch16', num_classes=2):
        super().__init__()
        clip = CLIPModel.from_pretrained(base)
        self.vision = clip.vision_model       # CLIP의 비전 인코더
        feat = clip.config.vision_config.hidden_size   # 768
        self.head = nn.Sequential(
            nn.LayerNorm(feat),
            nn.Dropout(0.1),
            nn.Linear(feat, num_classes)
        )
    def forward(self, x):
        out = self.vision(pixel_values=x)
        feat = out.pooler_output              # CLS token feature
        return self.head(feat)

model = CLIPDetector().to(device)
print('CLIP-ViT 백본 로드 완료')

In [ ]:
from transformers import CLIPModel

class CLIPDetector(nn.Module):
    """CLIP-ViT 백본 + 분류 헤드 (PEFT 없이)"""
    def __init__(self, base='openai/clip-vit-base-patch16', num_classes=2, dropout=0.1):
        super().__init__()
        clip = CLIPModel.from_pretrained(base)
        self.vision = clip.vision_model
        feat = clip.config.vision_config.hidden_size
        self.head = nn.Sequential(
            nn.LayerNorm(feat),
            nn.Dropout(dropout),
            nn.Linear(feat, num_classes)
        )

    def forward(self, x):
        out = self.vision(pixel_values=x)
        return self.head(out.pooler_output)

    def freeze_all_backbone(self):
        for p in self.vision.parameters(): p.requires_grad = False

    def unfreeze_last_n_blocks(self, n=2):
        """마지막 n개 Transformer 블록만 학습 가능하게"""
        for p in self.vision.parameters(): p.requires_grad = False
        for blk in self.vision.encoder.layers[-n:]:
            for p in blk.parameters(): p.requires_grad = True
        # post-layernorm도 같이
        if hasattr(self.vision, 'post_layernorm'):
            for p in self.vision.post_layernorm.parameters(): p.requires_grad = True

# 모델 생성
model = CLIPDetector(dropout=0.1).to(device)
model.unfreeze_last_n_blocks(n=2)   # ✅ 마지막 2 블록만 학습

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 {n_total:,}  |  학습 가능 {n_train:,}  ({100*n_train/n_total:.2f}%)')
# → CLIP-ViT-B/16: 약 86M 중 ~14M 학습 (약 16%)

In [ ]:
cls_w = compute_class_weight('balanced', classes=np.array([0,1]), y=train_meta['label'].values)
cls_w = torch.tensor(cls_w, dtype=torch.float).to(device)
print('클래스 가중치 (real, ai):', cls_w.cpu().numpy().round(3))

criterion = nn.CrossEntropyLoss(weight=cls_w)
scaler = GradScaler('cuda')

# ✅ 백본(작은 LR) + 헤드(큰 LR) 차등 LR
optimizer = torch.optim.AdamW([
    {'params':[p for p in model.vision.parameters() if p.requires_grad], 'lr': 1e-5},
    {'params': model.head.parameters(), 'lr': LR_HEAD},
], weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
def run_epoch(model, loader, train=True, opt=None):
    model.train(train)
    losses, preds, ys = [], [], []
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            with autocast('cuda'):
                logits = model(x); loss = criterion(logits, y)
            if train:
                opt.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(opt); scaler.update()
        losses.append(loss.item())
        preds.extend(torch.softmax(logits,1)[:,1].detach().cpu().numpy())
        ys.extend(y.cpu().numpy())
    f1 = f1_score(ys, np.array(preds)>0.5, zero_division=0)
    return np.mean(losses), f1

In [ ]:
hist = {'train_loss':[], 'val_loss':[], 'val_f1':[]}
best_f1 = 0

for ep in range(EPOCHS):
    tl, tf = run_epoch(model, train_loader, train=True,  opt=optimizer)
    vl, vf = run_epoch(model, val_loader,   train=False, opt=None)
    scheduler.step()
    hist['train_loss'].append(tl); hist['val_loss'].append(vl); hist['val_f1'].append(vf)
    print(f'[EP {ep+1:2d}/10] train_loss={tl:.4f} | val_loss={vl:.4f} | val_F1={vf:.3f}')
    if vf > best_f1:
        best_f1 = vf
        torch.save(model.state_dict(), CKPT/'best_initial.pt')
print(f'\n✅ Best val F1: {best_f1:.3f}')

# 학습 곡선
fig, ax = plt.subplots(1,2, figsize=(11,4))
ax[0].plot(hist['train_loss'], 'o-', label='Train'); ax[0].plot(hist['val_loss'], 's-', label='Val')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(hist['val_f1'], 'o-'); ax[1].set_title('Val F1'); ax[1].grid(True)
plt.tight_layout(); plt.savefig(OUT/'curve_initial.png', dpi=120); plt.show()

In [ ]:
def predict(model, loader, measure_time=True):
    model.eval()
    yt, yp, paths, times = [], [], [], []
    with torch.no_grad(), autocast('cuda'):
        for x, y, p in loader:
            x = x.to(device)
            if measure_time:
                # 1장당 시간 측정용
                t0 = time.time()
                logits = model(x)
                torch.cuda.synchronize()
                t = (time.time() - t0) / x.size(0)
                times.extend([t]*x.size(0))
            else:
                logits = model(x)
            prob = torch.softmax(logits,1)[:,1].cpu().numpy()
            yt.extend(y.numpy()); yp.extend(prob); paths.extend(p)
    return np.array(yt), np.array(yp), np.array(paths), np.array(times)

model.load_state_dict(torch.load(CKPT/'best_initial.pt'))
yt, yp, pths, times = predict(model, test_loader)
print(f'Test 샘플 {len(yt)}개  | 평균 추론 시간 {times.mean()*1000:.2f} ms/장')

In [ ]:
def evaluate_full(y_true, y_prob, infer_time_sec, threshold=0.5, title='초기 모델'):
    yhat = (y_prob > threshold).astype(int)
    cm = confusion_matrix(y_true, yhat, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    acc  = accuracy_score(y_true, yhat)
    prec = precision_score(y_true, yhat, zero_division=0)
    rec  = recall_score(y_true, yhat, zero_division=0)
    f1   = f1_score(y_true, yhat, zero_division=0)
    fpr  = fp / (fp + tn) if (fp+tn) > 0 else 0.0

    print(f'========== 성능 평가 결과 ({title}) ==========')
    print(f'Accuracy       : {acc*100:.2f}%')
    print(f'Precision      : {prec*100:.2f}%')
    print(f'Recall         : {rec*100:.2f}%  ← 핵심')
    print(f'F1 Score       : {f1*100:.2f}%')
    print(f'FPR            : {fpr*100:.2f}%')
    print(f'Inference Time : {infer_time_sec:.4f}초')
    print(f'------------------------------------')
    print(f'TP: {tp:2d}  FP: {fp:2d}  FN: {fn:2d}  TN: {tn:2d}')
    print(f'====================================')

    return {'accuracy':acc,'precision':prec,'recall':rec,
            'f1':f1,'fpr':fpr,'inference_time':infer_time_sec,
            'tp':int(tp),'fp':int(fp),'fn':int(fn),'tn':int(tn),'cm':cm}

result_initial = evaluate_full(yt, yp, times.mean(), title='초기 모델 (10 epoch)')

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ConfusionMatrixDisplay(result_initial['cm'], display_labels=['real','ai']).plot(
    ax=ax, cmap='Blues', colorbar=False)
plt.title(f"Confusion Matrix — Initial\nF1={result_initial['f1']*100:.2f}%")
plt.tight_layout(); plt.savefig(OUT/'cm_initial.png', dpi=120); plt.show()

In [ ]:
import optuna

def objective(trial):
    lr_back  = trial.suggest_float('lr_back',  1e-6, 1e-4, log=True)
    lr_head  = trial.suggest_float('lr_head',  1e-4, 1e-3, log=True)
    n_blocks = trial.suggest_categorical('n_blocks', [1, 2, 3])
    dropout  = trial.suggest_float('dropout', 0.0, 0.3)
    wd       = trial.suggest_float('wd', 1e-5, 1e-3, log=True)

    m = CLIPDetector(dropout=dropout).to(device)
    m.unfreeze_last_n_blocks(n=n_blocks)

    opt = torch.optim.AdamW([
        {'params':[p for p in m.vision.parameters() if p.requires_grad], 'lr': lr_back},
        {'params': m.head.parameters(), 'lr': lr_head},
    ], weight_decay=wd)

    best = 0
    for ep in range(5):
        run_epoch(m, train_loader, train=True, opt=opt)
        _, vf = run_epoch(m, val_loader, train=False)
        if vf > best: best = vf
        trial.report(vf, ep)
        if trial.should_prune(): raise optuna.TrialPruned()

    del m, opt
    torch.cuda.empty_cache()
    return best

study = optuna.create_study(direction='maximize',
                            pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=10, show_progress_bar=True)

print('\n🏆 Best params:', study.best_params)
print(f'🏆 Best val F1: {study.best_value:.3f}')

In [ ]:
bp = study.best_params

model2 = CLIPDetector(dropout=bp['dropout']).to(device)
model2.unfreeze_last_n_blocks(n=bp['n_blocks'])

opt2 = torch.optim.AdamW([
    {'params':[p for p in model2.vision.parameters() if p.requires_grad], 'lr': bp['lr_back']},
    {'params': model2.head.parameters(), 'lr': bp['lr_head']},
], weight_decay=bp['wd'])
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS)

hist2 = {'train_loss':[], 'val_loss':[], 'val_f1':[]}
best_f1_2 = 0
for ep in range(EPOCHS):
    tl, tf = run_epoch(model2, train_loader, train=True,  opt=opt2)
    vl, vf = run_epoch(model2, val_loader,   train=False, opt=None)
    sched2.step()
    hist2['train_loss'].append(tl); hist2['val_loss'].append(vl); hist2['val_f1'].append(vf)
    print(f'[Tuned EP {ep+1:2d}/10] train={tl:.4f} val={vl:.4f} F1={vf:.3f}')
    if vf > best_f1_2:
        best_f1_2 = vf
        torch.save(model2.state_dict(), CKPT/'best_tuned.pt')
print(f'\n✅ Tuned best val F1: {best_f1_2:.3f}')

In [ ]:
model2.load_state_dict(torch.load(CKPT/'best_tuned.pt'))
yt2, yp2, _, times2 = predict(model2, test_loader)
result_tuned = evaluate_full(yt2, yp2, times2.mean(), title='튜닝 모델 (Optuna)')

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ConfusionMatrixDisplay(result_tuned['cm'], display_labels=['real','ai']).plot(
    ax=ax, cmap='Greens', colorbar=False)
plt.title(f"Confusion Matrix — Tuned\nF1={result_tuned['f1']*100:.2f}%")
plt.tight_layout(); plt.savefig(OUT/'cm_tuned.png', dpi=120); plt.show()

In [ ]:
comp = pd.DataFrame({
    'Metric':         ['Accuracy','Precision','Recall(핵심)','F1','FPR','Inference Time(ms)'],
    'Initial':        [f'{result_initial["accuracy"]*100:.2f}%',
                       f'{result_initial["precision"]*100:.2f}%',
                       f'{result_initial["recall"]*100:.2f}%',
                       f'{result_initial["f1"]*100:.2f}%',
                       f'{result_initial["fpr"]*100:.2f}%',
                       f'{result_initial["inference_time"]*1000:.2f}'],
    'Tuned':          [f'{result_tuned["accuracy"]*100:.2f}%',
                       f'{result_tuned["precision"]*100:.2f}%',
                       f'{result_tuned["recall"]*100:.2f}%',
                       f'{result_tuned["f1"]*100:.2f}%',
                       f'{result_tuned["fpr"]*100:.2f}%',
                       f'{result_tuned["inference_time"]*1000:.2f}'],
})
print(comp.to_string(index=False))
comp.to_csv(OUT/'comparison.csv', index=False)

In [ ]:
import google.generativeai as genai
import google.api_core.exceptions as g_exc
from google.colab import userdata
import getpass

def get_key():
    k = None
    try:
        raw = userdata.get('GOOGLE_API_KEY')
        if isinstance(raw,str) and raw.startswith('AIza'): k = raw
        elif isinstance(raw,dict): k = raw.get('data',{}).get('payload') or raw.get('payload')
    except: pass
    if not k: k = getpass.getpass('GOOGLE_API_KEY: ').strip()
    return k

genai.configure(api_key=get_key())
GEMINI_MODEL = 'gemini-2.5-flash-lite'
print(f'✅ {GEMINI_MODEL} 준비 완료')

In [ ]:
LOCATE_PROMPT = """이 이미지가 AI 생성인지 실제인지 분석해주세요.
1) 먼저 의심스러운 영역(예: 손, 얼굴, 텍스트, 배경 등)을 식별하세요.
2) 그 영역에 어떤 이상이 있는지 짧게 설명하세요.
JSON으로만 출력:
{"suspicious_regions": ["region1", "region2"], "initial_verdict": "AI" or "real"}"""

EXAMINE_PROMPT = """앞서 의심 영역을 식별했습니다: {regions}
이제 그 영역들을 자세히 살펴보고 최종 판정을 내려주세요.
CNN 모델의 AI 확률: {prob:.2%}

JSON 스키마:
{{
  "verdict": "AI-generated" | "real",
  "confidence": 0.0~1.0,
  "evidence": [
    {{"region": "...", "artifact": "...", "severity": "high|mid|low"}}
  ],
  "summary_ko": "한국어 2~3문장 요약"
}}
JSON만 출력."""

def call_gemini(contents, max_retry=3):
    gm = genai.GenerativeModel(GEMINI_MODEL)
    for a in range(max_retry):
        try: return gm.generate_content(contents)
        except g_exc.TooManyRequests:
            time.sleep(15*(2**a))
    raise RuntimeError('Gemini 재시도 실패')

def detect_image_2stage(img_path, model, threshold=0.5):
    # CNN 1차 판정
    img = Image.open(img_path).convert('RGB')
    x = eval_tf(img).unsqueeze(0).to(device)
    with torch.no_grad(), autocast('cuda'):
        prob_ai = torch.softmax(model(x),1)[0,1].item()
    # Stage 1: Locate
    r1 = call_gemini([LOCATE_PROMPT, img])
    try:
        s1 = json.loads(r1.text.strip().lstrip('```json').rstrip('```').strip())
        regions = s1.get('suspicious_regions', [])
    except: regions = ['전체']
    # Stage 2: Examine
    r2 = call_gemini([EXAMINE_PROMPT.format(regions=regions, prob=prob_ai), img])
    try:
        s2 = json.loads(r2.text.strip().lstrip('```json').rstrip('```').strip())
    except Exception as e:
        s2 = {'raw': r2.text, 'parse_error': str(e)}
    return {'image': str(img_path), 'cnn_prob_ai': prob_ai,
            'locate': s1 if 'regions' not in s1 else s1, 'examine': s2}

In [ ]:
samples = test_meta.sample(3, random_state=1).to_dict('records')
results_xai = []
for i, r in enumerate(samples):
    print(f"\n🎬 [{i+1}] {Path(r['path']).name}  (GT={'AI' if r['label']==1 else 'real'})")
    out = detect_image_2stage(r['path'], model2)
    print(json.dumps(out, ensure_ascii=False, indent=2)[:1500])
    results_xai.append(out)
    if i < len(samples)-1: time.sleep(6)

In [ ]:
final = {
    'dataset': {'ai': len(ai_imgs), 'real': len(real_imgs),
                'split': {'train': len(train_meta), 'val': len(val_meta), 'test': len(test_meta)}},
    'model_initial':  {'arch':'CLIP-ViT-B/16 + LoRA', 'epochs': EPOCHS,
                       **{k:v for k,v in result_initial.items() if k != 'cm'}},
    'best_hyperparams': study.best_params,
    'model_tuned': {'epochs': EPOCHS,
                    **{k:v for k,v in result_tuned.items() if k != 'cm'}},
    'sample_xai': results_xai,
}
with open(OUT/'final_report.json','w',encoding='utf-8') as f:
    json.dump(final, f, ensure_ascii=False, indent=2, default=str)
print('✅ 저장 완료')

import zipfile
zp = '/content/results.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'): z.write(p, p.relative_to(OUT))
from google.colab import files
files.download(zp)